# ModernBERT RAG (Rebalanced Dataset)


# Setup

In [1]:
import datasets
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
from google.colab import drive, userdata
from huggingface_hub import login, hf_hub_download

cache_path = "/content/huggingface_cache"
os.makedirs(cache_path, exist_ok=True)
os.environ['HF_HOME'] = cache_path

if userdata.get('HF_TOKEN'):
  login(token=userdata.get('HF_TOKEN'))
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="utils.py", repo_type="dataset",local_dir=".")
  hf_hub_download(repo_id="sookiemonster/asrs-narratives", filename="all_contributing_factors.csv", repo_type="dataset",local_dir=".")

raw_dataset = datasets.load_dataset("sookiemonster/asrs-narratives")
factors = pd.read_csv("all_contributing_factors.csv", index_col=0)

utils.py: 0.00B [00:00, ?B/s]

all_contributing_factors.csv: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/9.74M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/10360 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [2]:
train_ds = datasets.load_dataset("sookiemonster/asrs-narratives-rebalance", split='train')
valid_ds = raw_dataset['validation']
test_ds = raw_dataset['test']

labels = train_ds.features['label'].names

id_to_label = { idx : label for idx, label in enumerate(labels) }
label_to_id = { label : idx for idx, label in id_to_label.items() }

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/22.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/4.09M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/9.38M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/22992 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/4441 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/9868 [00:00<?, ? examples/s]

In [3]:
from functools import partial

def filter_labels(ds, to_remove:list):
  to_remove_set = set(to_remove)
  return ds.filter(lambda example : id_to_label[example['label']] not in to_remove_set)

filter_ambiguous = partial(filter_labels, to_remove=['ambiguous'])

filtered_train_ds = filter_ambiguous(train_ds)
filtered_valid_ds = filter_ambiguous(valid_ds)

Filter:   0%|          | 0/22992 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4441 [00:00<?, ? examples/s]

In [4]:
def _validate_groupings(groupings:dict[str, set]):
  mut_excl = [va.isdisjoint(vb) for ka, va in groupings.items() for kb, vb in groupings.items() if ka != kb]
  assert all(mut_excl), f"{mut_excl}"

  all_labels = set([label for val_set in groupings.values() for label in val_set])
  assert all_labels == set(id_to_label.values()), f"Missing: {set(id_to_label.values()) - all_labels}"


def group_labels(ds, groupings:dict[str, set]):
  _validate_groupings(groupings)
  group_names = list(groupings.keys())
  group_names.sort()

  fine_grained_label_to_group = {
      label : group_name for group_name, val_set in groupings.items() for label in val_set
  }

  res = ds.map(lambda ex: {"group" : fine_grained_label_to_group[ id_to_label[ex['label']] ]})
  res = res.filter(lambda ex: ex["group"] != 'DELETE')

  new_features = res.features.copy()
  group_names.remove("DELETE")
  new_features["group"] = ClassLabel(names=group_names)

  res = res.cast(new_features)
  return res

## Setup Dataset


In [5]:
from datasets import ClassLabel

groupings = {
    'DELETE' : set(['ambiguous']),
}

groupings = groupings | {
    label : set([label]) for label in id_to_label.values() if label != 'ambiguous'
}

grouped_ds_train = group_labels(filtered_train_ds, groupings)
grouped_ds_valid = group_labels(filtered_valid_ds, groupings)

Map:   0%|          | 0/22157 [00:00<?, ? examples/s]

Filter:   0%|          | 0/22157 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/22157 [00:00<?, ? examples/s]

Map:   0%|          | 0/4083 [00:00<?, ? examples/s]

Filter:   0%|          | 0/4083 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/4083 [00:00<?, ? examples/s]

In [6]:
id_to_group = { i: group for i, group in enumerate(grouped_ds_train.features['group'].names) }
id_to_group

{0: 'aircraft',
 1: 'airport',
 2: 'airspacestructure',
 3: 'atcequipment/navfacility/buildings',
 4: 'chartorpublication',
 5: 'companypolicy',
 6: 'environment-nonweatherrelated',
 7: 'equipment/tooling',
 8: 'humanfactors',
 9: 'incorrect/notinstalled/unavailablepart',
 10: 'logbookentry',
 11: 'manuals',
 12: 'mel',
 13: 'procedure',
 14: 'softwareandautomation',
 15: 'staffing',
 16: 'weather'}

# Inference

## Install Flash-Attention

In [7]:
import torch
import sys

print(f"Python Version: {sys.version_info.major}.{sys.version_info.minor}")
print(f"PyTorch Version: {torch.__version__.split('+')[0]}")
print(f"CUDA Version (Torch): {torch.version.cuda}")
print(f"CXX11 ABI: {torch._C._GLIBCXX_USE_CXX11_ABI}")

Python Version: 3.12
PyTorch Version: 2.10.0
CUDA Version (Torch): 12.8
CXX11 ABI: True


In [ ]:
!pip install "https://github.com/lesj0610/flash-attention/releases/download/v2.8.3-cu12-torch2.10-cp312/flash_attn-2.8.3%2Bcu12torch2.10cxx11abiTRUE-cp312-cp312-linux_x86_64.whl"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 253.6/253.6 MB 5.1 MB/s eta 0:00:00


## Run Model

In [8]:
from sentence_transformers import SentenceTransformer

tuned_model = "sookiemonster/asrs-modernbert-rebalance-weighted"
model = SentenceTransformer(tuned_model)

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/299M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

ModernBertModel LOAD REPORT from: sookiemonster/asrs-modernbert-rebalance-weighted
Key               | Status     |  | 
------------------+------------+--+-
head.norm.weight  | UNEXPECTED |  | 
classifier.weight | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 
classifier.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
sentences = list(grouped_ds_train['text'])
print(len(sentences))
sentences[:5]

22157


["Narrative 1 - 'I was working ZZZ and North arrival. I had been on position for about 20 minutes and there was weather on the scope when I sat down. I was briefed that there had been some deviations but just light chop. And area of heavy precipitation which had been west of ZZZ1 eventually moved onto the final but aircraft were still taking the approach and landing. Eventually we got some LLWS alerts which I issued which in short time were followed by several go arounds and a microburst alert which I also issued as well as described the precipitation along the final. Aircraft X said he needed to discontinue the approach so I issued a 220 heading and 5000 feet since departure was getting a go-around in front of him and I couldn't turn him left or right due to airspace constraints. I yelled to departure and shipped him. In the mean time every other aircraft on final clogged up the frequency saying they also wouldn't take the approach. At some point Aircraft X said that he encountered al

In [ ]:
embeddings = model.encode(sentences)
np.save("all_rebalance_train_embeddings.npy", embeddings)

W0501 13:49:35.918000 15956 torch/_inductor/utils.py:1679] [1/0_1] Not enough SMs to use max_autotune_gemm mode


In [ ]:
embeddings.shape

(22157, 768)

##

# Load & Index Embeddings

In [9]:
!pip install faiss-gpu-cu12
# !pip install faiss-cpu

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 MB 15.0 MB/s eta 0:00:00


In [11]:
embeddings = np.load('all_rebalance_train_embeddings.npy')

In [12]:
# Normalize to unit vectors
row_norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
norm_embed = embeddings / row_norms
norm_embed.shape

(22157, 768)

In [15]:
train_df = grouped_ds_train.to_pandas().set_index('acn')
train_df.index = train_df.index.astype('int')
train_df['group'] = train_df['group'].map(id_to_group)
group_masks = train_df.groupby('group').indices

In [18]:
import faiss
import numpy as np

dimension = norm_embed.shape[1]
index = faiss.IndexFlatIP(dimension)
index.add(norm_embed.astype('float32'))

def search_rag(query_text, k=5):
    query_vector = model.encode([query_text], convert_to_numpy=True).astype('float32')

    distances, indices = index.search(query_vector, k)
    results = train_df.iloc[indices[0]]
    return results

In [29]:
def find_similar(query_vector, cat_k=2, min_similarity=0.7):
  """
  Use cat_k=-1 to include all categories (and get similarities)
  """
  results = []
  if cat_k == -1:
    lims, scores, indices = index.range_search(query_vector, thresh=min_similarity)

    i = 0
    for acn, row in train_df.iloc[indices].iterrows():
      results.append([acn, row['text'], row['group'], scores[i]])
      i += 1

    df= pd.DataFrame(results, columns=['acn', 'text', 'label', 'score'])
    return df.sort_values(by=['score'], ascending=False)


  for cat, mask in group_masks.items():
    selector = faiss.IDSelectorArray(mask.astype('int64'))
    params = faiss.SearchParameters(sel=selector)
    scores, indices = index.search(query_vector, cat_k, params=params)

    is_similar = scores[0] >= min_similarity
    relevant_indices = indices[0][is_similar]
    relevant_scores = scores[0][is_similar]

    if not any(is_similar):
      continue

    for idx, text in enumerate(train_df.iloc[relevant_indices]['text'].values):
      results.append([cat, text, relevant_scores[idx]])

  return pd.DataFrame(results, columns=['label', 'text', 'score'])

In [32]:
res = find_similar(np.array([norm_embed[24]]), cat_k=-1, )
res

,acn,text,label,score
0,2100261,Narrative 1 - 'The day started by working a la...,humanfactors,1.000000
3,1643427,Narrative 1 - 'At rotation the flight deck doo...,companypolicy,0.748866
1,1791930,Narrative 1 - 'During brief; maintenance write...,humanfactors,0.728330
4,1768223,Narrative 1 - 'On our third and final flight a...,humanfactors,0.708821
2,2233308,Narrative 1 - 'We received a forward entry doo...,aircraft,0.708697


In [20]:
grouped_ds_train[6]

{'acn': 1853327,
 'text': "Narrative 1 - 'While on ILS final to LAX Runway 25L at approximately 7 miles from Runway 25L and 2;000 feet AGL; LAX Tower cleared a B747 to take off in front of us on 25L. Both the Captain and I remarked that we were unhappy with the spacing and while it looked like the runway would be clear in time; we both remarked that we should be prepared to go-around. The Captain elected to fly slightly above the glide path to avoid turbulence; but at around 200 feet AGL we encountered moderate turbulence from the departing B747 lasting around 1-2 seconds. After exiting the turbulence I estimated we would touch down 2;000-2;500 feet down; and I said; 'We're going to be long.' The Captain made a smooth correction to glide path and a smooth touchdown probably around the 2;000 foot point. There were no GPWS warnings or 'long landing' warnings at any point. I don't know whether the Tower spacing between us and the B747 met standards; but I thought it was a poor decision to

In [21]:
res = find_similar(np.array([norm_embed[6]]), min_similarity=0.7, cat_k=1)
res

,label,text
0,environment-nonweatherrelated,Narrative 1 - 'After on speed; normal sink rat...
1,humanfactors,Narrative 1 - 'We were flying the ILS 27R to O...
2,procedure,Narrative 1 - 'While on ILS final to LAX Runwa...


In [ ]:
processed_to_unprocessed = {
    'airport': 'Airport',
    'airspacestructure': 'Airspace Structure',
    'atcequipment/navfacility/buildings': 'ATC Equipment / Nav Facility / Buildings',
    'chartorpublication': 'Chart or Publication',
    'companypolicy': 'Company Policy',
    'environment-nonweatherrelated': 'Environment - Non-weather Related',
    'equipment/tooling': 'Equipment / Tooling',
    'incorrect/notinstalled/unavailablepart': 'Incorrect / Not Installed / Unavailable Part',
    'logbookentry': 'Logbook Entry',
    'manuals': 'Manuals',
    'mel': 'MEL (Minimum Equipment List)',
    'softwareandautomation': 'Software and Automation',
    'staffing': 'Staffing'
}

template = """Prelabeled Incident: {text}
Existing Analyst Label: {label}
---
"""
def format_examples(df: pd.DataFrame):
  res = "\n"
  for idx, row in df.iterrows():
    res += template.format(text=row['text'], label=processed_to_unprocessed[row['label']])
  return res

print(format_examples(res))


Prelabeled Incident: Narrative 1 - 'On approach to [Runway] 34R at DEN and descending below 10;000; we had a total lost of our GPS signal.  We did not get it back the entire approach and landing. We notified ATC. We were on vectors to the ILS so it did not affect our flight. DEN ATC knew about the issue and were not issuing any clearances for RNAV approaches. Multiple aircraft were having the same issue. We also had an ADS-B reporting fault at the same time.'
Existing Analyst Label: Airspace Structure
--- 
Prelabeled Incident: Narrative 1 - 'Due to continued 5G interference in the DEN area we lost GPS-L; GPS-R; and got an XPNDER FAIL light turning over FFFAT on the LAWGR 3 arrival.  Later after capturing the localizer to 35R we could see the depicted path on the navigation display was offset from the actual localizer final approach course.  No further issue or anomaly was experienced.  Systems returned to normal by the time we parked at the gate.'
Existing Analyst Label: Airspace Stru